# Agentic RAG, OpenAI Agents SDK, MCP

My 3 favorite things in 12 minutes.

Please see the README for setup instructions.

### Video series:

1. Comparing Agentic RAG with traditional RAG:  
https://youtu.be/K6wpRkJrcpM

2. THIS VIDEO: Building the initial version of this:  
https://youtu.be/KdKqMfs-8gs

3. Taking Agentic RAG to the next level with more tools.   
https://youtu.be/UmGLirGbKTk  

In [1]:
from agents import Agent, Runner, SQLiteSession
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
import os
from IPython.display import display, Markdown
from pathlib import Path
from qdrant_client import QdrantClient
from agents.extensions.models.litellm_model import LitellmModel
import gradio as gr
load_dotenv(override=True)

True

## Picking your LLM provider, including Cerebras

If you'd like to use Cerebras, the high speed inference provider, with open-source model gpt-oss-120b, then sign up for a Cerebras account here:  
https://cloud.cerebras.ai/

Alternatively, to use OpenAI models like gpt-5.4-mini, replace the entire contents of the next cell with:

```python
model = "gpt-5.4-mini"
```

You can also use OpenRouter:

```python
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
model = LitellmModel(model="openrouter/openrouter-model-name", api_key=openrouter_api_key)
```


In [2]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
model = LitellmModel(model="openrouter/openai/gpt-oss-120b:free", api_key=openrouter_api_key)

In [15]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
model = LitellmModel(model="openrouter/qwen/qwen3.6-plus", api_key=openrouter_api_key)


In [3]:
agent = Agent("Tester", model=model)
response = await Runner.run(agent, "what is 2+2?")
print(response.final_output)

2 + 2 = 4.


## Which URLs to examine for content for our Agent?

Here I specify some of my websites. You should provide whichever have the information that you want in your agent's memory. I put the same URL in the list multiple times so that the agent stores more memories.

In [4]:
urls = ["https://edwarddonner.com", "https://edwarddonner.com/about"] * 3 + ["https://edwarddonner.com/curriculum"] * 10
urls

['https://edwarddonner.com',
 'https://edwarddonner.com/about',
 'https://edwarddonner.com',
 'https://edwarddonner.com/about',
 'https://edwarddonner.com',
 'https://edwarddonner.com/about',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum',
 'https://edwarddonner.com/curriculum']

## The MCP Parameters

In [5]:
knowledge_dir = Path.cwd() / "knowledge"
knowledge_dir.mkdir(exist_ok=True)
vectordb_path = knowledge_dir / "vectordb"

fetch_params = {
    "command": "uvx",
    "args": ["mcp-server-fetch"],
}

vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

In [6]:
CONTEXT = """
You are an Agent with expert knowledge about Ed Donner with particular focus on his online AI courses.
"""

INSTRUCTIONS = CONTEXT + """
You are populating your memories with information retrieved from a given website.
Use your MCP tools to retrieve the website. Extract key knowledge. Check what's already in your memories to avoid duplicates.
After you are done, reply with a brief status update and the number of memories you added.
Aim to add at least 10 unique memories, unless your existing memories are already comprehensive.
"""

In [ ]:
async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=120) as fetch_mcp:
    tools = await fetch_mcp.list_tools()
tools

UnsupportedOperation: fileno

In [ ]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
    tools = await vectorstore_mcp.list_tools()
tools

## The Ingest Agent

In [ ]:
for url in urls:
    async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=120) as fetch_mcp:
        async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
            agent = Agent(name="Ingester", model=model, instructions=INSTRUCTIONS, mcp_servers=[fetch_mcp, vectorstore_mcp])
            task = f"Add unique memories with information from this website: {url} and reply with a one sentence status update including how many memories were added."
            response = await Runner.run(agent, task, max_turns=50)
            display(Markdown(response.final_output))

In [ ]:
client = QdrantClient(path=str(vectordb_path))
collection_name = "knowledge"

info = client.get_collection(collection_name)
print(f"Memories in '{collection_name}': {info.points_count}\n")

points, _ = client.scroll(collection_name=collection_name, limit=200, with_payload=True, with_vectors=False)
for i, p in enumerate(points, 1):
    doc = (p.payload or {}).get("document", "")
    preview = doc.replace("\n", " ")[:160]
    print(f"{i:>3}. {preview}{'...' if len(doc) > 160 else ''}")

client.close()

In [ ]:
EXPERT_INSTRUCTIONS = """
You are an expert about Ed Donner and his online courses. You are answering questions about him and his courses to visitors on his website.
Use your memories to help answer the question. If you don't know the answer, say so.
"""

EXAMPLES = ["Which course covers RAG?", "Which course covers MCP?", "How do I become an expert on Claude Code?"]

In [ ]:
convo = SQLiteSession("test_conversation")

## Agentic RAG + OpenAI Agents SDK + MCP in a 4 line function!

In [ ]:
async def chat(message, history):
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
        agent = Agent(name="Expert", model=model, instructions=EXPERT_INSTRUCTIONS, mcp_servers=[vectorstore_mcp])
        response = await Runner.run(agent, message, session=convo)
        return response.final_output

In [ ]:
from styles import CSS, JS
gr.ChatInterface(chat, examples=EXAMPLES, chatbot=gr.Chatbot(show_label=False, height=700)).launch(css=CSS, js=JS, theme=gr.themes.Base(), inbrowser=True)

### Video series:

1. Comparing Agentic RAG with traditional RAG:  
https://youtu.be/K6wpRkJrcpM

2. THIS VIDEO: Building the initial version of this:  
https://youtu.be/KdKqMfs-8gs

3. Taking Agentic RAG to the next level with more tools.   
https://youtu.be/UmGLirGbKTk  